# AutoDSPy GPT-2 / GRPO-inspired / GSM8K / Llama 3.1 8B reproduction

Isolated from the unchanged original notebook. Read docs/REPRODUCTION_AUDIT.md and docs/DEVIATIONS_FROM_PAPER.md first. The default follows the released simplified GRPO loss, not clipped textbook GRPO. No cleanup cell or implicit checkpoint loading. Full training is explicitly opt-in.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import sys, subprocess
from pathlib import Path
from autodspy_reproduction.config import load_config
ROOT = Path.cwd()
assert (ROOT / "configs/grpo_gsm8k.yaml").exists(), "Start notebook from repository root"
SMOKE = "configs/grpo_smoke_test.yaml"
FULL = "configs/grpo_gsm8k.yaml"
full_config = load_config(FULL)
smoke_config = load_config(SMOKE)
print(full_config.to_dict())

## Validation

Use the repository environment as this notebook kernel. These tests use explicitly labeled fixtures and do not claim Llama benchmark results.

In [ ]:
def cli(command, config=SMOKE, *extra):
    return subprocess.run([sys.executable, "-m", "autodspy_reproduction", command, "--config", config, *extra], cwd=ROOT, check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py", "-v"], cwd=ROOT, check=True)

## Live execution probe and smoke

Start Ollama and install llama3.1:8b on the execution host. For a remote endpoint set EXTRA to ["--endpoint", "http://HOST:11434"] consistently. CPU Llama is the YAML default; GPT-2 trains on CUDA. Smoke uses one episode, two prompts, K=5 and the real target models. Inspect the printed run directory and docs/VALIDATION_RESULTS.md.

In [ ]:
EXTRA = []
RUN_EXECUTION = False
RUN_SMOKE = False
if RUN_EXECUTION:
    cli("preflight", SMOKE, *EXTRA)
    cli("execute", SMOKE, *EXTRA)
if RUN_SMOKE:
    cli("train", SMOKE, *EXTRA)

## Full training

200 episodes x 200 prompt draws x 5 candidates = 200,000 pipeline evaluations plus possible judge/adapter calls. Sampling scope and update cadence are documented assumptions. Run only after reviewing a successful live smoke and its resource measurements.

In [ ]:
RUN_FULL = False
RESUME_CHECKPOINT = None
if RUN_FULL:
    resume_args = ["--resume", str(RESUME_CHECKPOINT)] if RESUME_CHECKPOINT else []
    cli("train", FULL, *EXTRA, *resume_args)

## Four-way held-out evaluation

Choose the same YAML and execution settings as the checkpoint. Full configuration evaluates all 1,319 official test cases; smoke configuration evaluates two seeded cases. Static Predict, static CoT, initial policy and trained policy share all execution/metric settings. Results are separate normalized exact-match and LLM-fallback accuracies, not the appendix embedding metric.

In [ ]:
RUN_EVALUATION = False
EVALUATION_CONFIG = SMOKE
CHECKPOINT = None
if RUN_EVALUATION:
    assert CHECKPOINT is not None, "Set an actual isolated reproduction checkpoint path"
    cli("evaluate", EVALUATION_CONFIG, *EXTRA, "--checkpoint", str(CHECKPOINT))

## Inspect actual artifacts

Never treat scripted engineering smoke rewards as benchmark results. Publication claims require completed full training, real held-out evaluation and resolution/disclosure of protocol ambiguities. Memory telemetry in metrics.jsonl is the policy process only; monitor total-device usage with nvidia-smi.

In [ ]:
import json
RUN_DIRECTORY = None
if RUN_DIRECTORY:
    run = Path(RUN_DIRECTORY)
    print(json.loads((run / "status.json").read_text()))
    if (run / "metrics.jsonl").exists():
        print((run / "metrics.jsonl").read_text())
    if (run / "results.json").exists():
        print(json.loads((run / "results.json").read_text()))